<a href="https://colab.research.google.com/github/Fabian-lewis/remote-work-productivity-analysis/blob/main/01_data_cleaning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [10]:
## Import Libraries
import pandas as pd
import numpy as np

In [11]:
## Load Data From Drive
from google.colab import drive
drive.mount('/content/drive')

# Import from drive
data_url = '/content/drive/MyDrive/NSW+Remote+Working+Survey'

## Load 2020 data
try:
  data_2020 = pd.read_csv(f'{data_url}/2020_rws.csv', encoding='windows-1252')
  print(f"2020 data is Loaded ☑ Shape: {data_2020.shape}")
except Exception as e:
  print("Failed to load data ❌")
  print(e)

from google.colab import files
from io import BytesIO

## Import from local machine
uploaded = files.upload()

## Load 2021 data
try:
  data_2021 = pd.read_csv(BytesIO(list(uploaded.values())[0]), encoding='utf-8')
  print(f"2021 data is Loaded ☑, Shape: {data_2021.shape}")
except Exception as e:
  print("Failed to load data ❌")
  print(e)



Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
2020 data is Loaded ☑ Shape: (1507, 73)


Saving 2021_rws - 2020_rws.csv to 2021_rws - 2020_rws.csv
2021 data is Loaded ☑, Shape: (1507, 73)


In [12]:
## Check if columns names are the same
same_columns = set(data_2020.columns) == set(data_2021.columns)

print("Are column names identical? ☑" if same_columns else "Column names differ ❌")


Are column names identical? ☑


In [13]:
## Print out the different columns names
only_2020 = set(data_2020.columns) - set(data_2021.columns)
only_2021 = set(data_2021.columns) - set(data_2020.columns)

print(f"Columns only in 2020 ({len(only_2020)}):")
for col in sorted(only_2020):
    print("  -", col)

print(f"\nColumns only in 2021 ({len(only_2021)}):")
for col in sorted(only_2021):
    print("  -", col)

Columns only in 2020 (0):

Columns only in 2021 (0):


In [14]:
## Renaming columns Pipeline

CORE_DEMOGRAPHICS_RENAME_MAP = {
    "Response ID": "response_id",
    "What year were you born?": "birth_year",
    "What is your gender?": "gender",
    "Which of the following best describes your industry?": "industry_broad",
    "Which of the following best describes your industry? (Detailed)": "industry_detailed",
    "Which of the following best describes your current occupation?": "occupation_broad",
    "Which of the following best describes your current occupation? (Detailed)": "occupation_detailed",
    "How many people are currently employed by your organisation?": "org_size",
    "Do you manage people as part of your current occupation?": "is_manager",
    "Which of the following best describes your household?": "household_type",
    "How long have you been in your current job?": "job_tenure",
    "Metro / Regional": "location_type"
}

REMOTE_WORK_INTENSITY_RENAME_MAP = {
    "Thinking about your current job, how much of your time did you spend remote working last year?": "last_year_remote_share_actual",
    "How much of your time would you have preferred to work remotely last year?": "last_year_remote_share_preferred",
    "Thinking about your current job, how much of your time did you spend remote working in the last 3 months?": "remote_share_recent_actual",
    "How much of your time would you have preferred to work remotely in the last 3 months?": "remote_share_recent_preferred",
    "Imagine that COVID-19 is cured or eradicated. Going forward, how much of your time would you prefer to work remotely?": "remote_share_future_preferred"
}

ORG_CULTURE_REMOTE_RENAME_MAP = {
    "Thinking about remote working last year, how strongly do you agree or disagree with the following statements? - My organisation encouraged people to work remotely": "org_encourage_remote",
    "Thinking about remote working last year, how strongly do you agree or disagree with the following statements? - My organisation was well prepared for me to work remotely": "org_prepared_remote",
    "Thinking about remote working last year, how strongly do you agree or disagree with the following statements? - It was common for people in my organisation to work remotely": "org_remote_common",
    "Thinking about remote working last year, how strongly do you agree or disagree with the following statements? - It was easy to get permission to work remotely": "org_permission_easy",
    "Thinking about remote working last year, how strongly do you agree or disagree with the following statements? - I could easily collaborate with colleagues when working remotely": "collab_easy_remote",
    "Thinking about remote working last year, how strongly do you agree or disagree with the following statements? - I would recommend remote working to others": "recommend_remote"
}

ORG_CULTURE_REMOTE_RENAME_MAP_RECENT = {
    "Thinking about remote working in the last 3 months, how strongly do you agree or disagree with the following statements? - My organisation encouraged people to work remotely": "org_encourage_remote_recent",
    "Thinking about remote working in the last 3 months, how strongly do you agree or disagree with the following statements? - My organisation was well prepared for me to work remotely": "org_prepared_remote_recent",
    "Thinking about remote working in the last 3 months, how strongly do you agree or disagree with the following statements? - It was common for people in my organisation to work remotely": "org_remote_common_recent",
    "Thinking about remote working in the last 3 months, how strongly do you agree or disagree with the following statements? - It was easy to get permission to work remotely": "org_permission_easy_recent",
    "Thinking about remote working in the last 3 months, how strongly do you agree or disagree with the following statements? - I could easily collaborate with colleagues when working remotely": "collab_easy_remote_recent",
    "Thinking about remote working in the last 3 months, how strongly do you agree or disagree with the following statements? - I would recommend remote working to others": "recommend_remote_recent"
}

FUTURE_EXPECTATIONS_RENAME_MAP = {
    "Imagine that COVID-19 is cured or eradicated.  How likely would you consider the following statements? - My employer would encourage more remote working": "future_org_encourage_remote_likelihood",
    "Imagine that COVID-19 is cured or eradicated.  How likely would you consider the following statements? - My employer would make changes to support remote working": "future_org_support_changes_likelihood",
    "Imagine that COVID-19 is cured or eradicated.  How likely would you consider the following statements? - I would have more choice about whether I work remotely": "future_remote_choice_likelihood"
}

PRODUCTIVITY_RENAME_MAP = {
    "This question is about your productivity. Productivity means what you produce for each hour that you work. It includes the amount of work you achieve each hour, and the quality of your work each hour.  \nPlease compare your productivity when you work remotely to when you work at your employer’s workplace.  \nRoughly how productive are you, each hour, when you work remotely?": "remote_productivity_vs_office"
}

TIME_USE_OFFICE_RENAME_MAP = {
    "On a day when you attend your employer's workplace, how many hours would you spend doing the following activities? - Preparing for work and commuting": "office_hours_commute",
    "On a day when you attend your employer's workplace, how many hours would you spend doing the following activities? - Working": "office_hours_work",
    "On a day when you attend your employer's workplace, how many hours would you spend doing the following activities? - Personal and family time": "office_hours_personal",
    "On a day when you attend your employer's workplace, how many hours would you spend doing the following activities? - Caring and domestic responsibilities": "office_hours_care_domestic"
}

TIME_USE_REMOTE_RENAME_MAP = {
    "On a day when you do remote work, how many hours would you spend doing the following activities? - Preparing for work and commuting": "remote_hours_commute",
    "On a day when you do remote work, how many hours would you spend doing the following activities? - Working": "remote_hours_work",
    "On a day when you do remote work, how many hours would you spend doing the following activities? - Personal and family time": "remote_hours_personal",
    "On a day when you do remote work, how many hours would you spend doing the following activities? - Caring and domestic responsibilities": "remote_hours_care_domestic"
}

MOST_SIGNIFICANT_BARRIERS_RENAME_MAP = {
    "From the following, please select the most significant barrier to doing your work remotely - Connectivity (internet connection) ; Feeling left out and/or isolated ; Poor management ; IT equipment (computer, printer, etc.) ; Difficulty collaborating remotely ; Caring responsibilities": "most_sig_barrier_1",
    "From the following, please select the most significant barrier to doing your work remotely - Connectivity (internet connection) ; Feeling left out and/or isolated ; Poor management ; Cyber security ; Lack of motivation ; Lack of motivation": "most_sig_barrier_2",
    "From the following, please select the most significant barrier to doing your work remotely - Connectivity (internet connection) ; Feeling left out and/or isolated ; Poor management ; My organisation's software and systems ; My workspace (e.g. suitable chair, lighting, noise levels, facilities) ; I have tasks that can't be done remotely": "most_sig_barrier_3",
    "From the following, please select the most significant barrier to doing your work remotely - Connectivity (internet connection) ; Feeling left out and/or isolated ; Poor management ; Lack of remote working skills ; My living situation (e.g. location, home size, who I live with) ; Management discourages remote working": "most_sig_barrier_4",
    "From the following, please select the most significant barrier to doing your work remotely - IT equipment (computer, printer, etc.) ; Difficulty collaborating remotely ; Caring responsibilities ; Cyber security ; Lack of motivation ; Lack of motivation": "most_sig_barrier_5",
    "From the following, please select the most significant barrier to doing your work remotely - IT equipment (computer, printer, etc.) ; Difficulty collaborating remotely ; Caring responsibilities ; My organisation's software and systems ; My workspace (e.g. suitable chair, lighting, noise levels, facilities) ; I have tasks that can't be done remotely": "most_sig_barrier_6",
    "From the following, please select the most significant barrier to doing your work remotely - IT equipment (computer, printer, etc.) ; Difficulty collaborating remotely ; Caring responsibilities ; Lack of remote working skills ; My living situation (e.g. location, home size, who I live with) ; Management discourages remote working": "most_sig_barrier_7",
    "From the following, please select the most significant barrier to doing your work remotely - Cyber security ; Lack of motivation ; Lack of motivation ; My organisation's software and systems ; My workspace (e.g. suitable chair, lighting, noise levels, facilities) ; I have tasks that can't be done remotely": "most_sig_barrier_8",
    "From the following, please select the most significant barrier to doing your work remotely - Cyber security ; Lack of motivation ; Lack of motivation ; Lack of remote working skills ; My living situation (e.g. location, home size, who I live with) ; Management discourages remote working": "most_sig_barrier_9",
    "From the following, please select the most significant barrier to doing your work remotely - My organisation's software and systems ; My workspace (e.g. suitable chair, lighting, noise levels, facilities) ; I have tasks that can't be done remotely ; Lack of remote working skills ; My living situation (e.g. location, home size, who I live with) ; Management discourages remote working": "most_sig_barrier_10",
}

LEAST_SIGNIFICANT_BARRIERS_RENAME_MAP = {
    "From the following, please select the least significant barrier to doing your work remotely - Connectivity (internet connection) ; Feeling left out and/or isolated ; Poor management ; IT equipment (computer, printer, etc.) ; Difficulty collaborating remotely ; Caring responsibilities": "least_sig_barrier_1",
    "From the following, please select the least significant barrier to doing your work remotely - Connectivity (internet connection) ; Feeling left out and/or isolated ; Poor management ; Cyber security ; Lack of motivation ; Lack of motivation": "least_sig_barrier_2",
    "From the following, please select the least significant barrier to doing your work remotely - Connectivity (internet connection) ; Feeling left out and/or isolated ; Poor management ; My organisation's software and systems ; My workspace (e.g. suitable chair, lighting, noise levels, facilities) ; I have tasks that can't be done remotely": "least_sig_barrier_3",
    "From the following, please select the least significant barrier to doing your work remotely - Connectivity (internet connection) ; Feeling left out and/or isolated ; Poor management ; Lack of remote working skills ; My living situation (e.g. location, home size, who I live with) ; Management discourages remote working": "least_sig_barrier_4",
    "From the following, please select the least significant barrier to doing your work remotely - IT equipment (computer, printer, etc.) ; Difficulty collaborating remotely ; Caring responsibilities ; Cyber security ; Lack of motivation ; Lack of motivation": "least_sig_barrier_5",
    "From the following, please select the least significant barrier to doing your work remotely - IT equipment (computer, printer, etc.) ; Difficulty collaborating remotely ; Caring responsibilities ; My organisation's software and systems ; My workspace (e.g. suitable chair, lighting, noise levels, facilities) ; I have tasks that can't be done remotely": "least_sig_barrier_6",
    "From the following, please select the least significant barrier to doing your work remotely - IT equipment (computer, printer, etc.) ; Difficulty collaborating remotely ; Caring responsibilities ; Lack of remote working skills ; My living situation (e.g. location, home size, who I live with) ; Management discourages remote working": "least_sig_barrier_7",
    "From the following, please select the least significant barrier to doing your work remotely - Cyber security ; Lack of motivation ; Lack of motivation ; My organisation's software and systems ; My workspace (e.g. suitable chair, lighting, noise levels, facilities) ; I have tasks that can't be done remotely": "least_sig_barrier_8",
    "From the following, please select the least significant barrier to doing your work remotely - Cyber security ; Lack of motivation ; Lack of motivation ; Lack of remote working skills ; My living situation (e.g. location, home size, who I live with) ; Management discourages remote working": "least_sig_barrier_9",
    "From the following, please select the least significant barrier to doing your work remotely - My organisation's software and systems ; My workspace (e.g. suitable chair, lighting, noise levels, facilities) ; I have tasks that can't be done remotely ; Lack of remote working skills ; My living situation (e.g. location, home size, who I live with) ; Management discourages remote working": "least_sig_barrier_10",
}

BEST_REMOTE_WORKING_ASPECT_RENAME_MAP = {
    "Compare remote working to working at your employer’s workplace. Select the best aspect of remote working for you - Managing my family responsibilities ; My working relationships ; Preparing for work and commuting ; The number of hours  I work ; My work-life balance ; My on-the-job learning opportunities": "best_aspect_1",
    "Compare remote working to working at your employer’s workplace. Select the best aspect of remote working for you - Managing my family responsibilities ; My working relationships ; Preparing for work and commuting ; Managing my personal commitments ; My opportunities to socialise ; My mental wellbeing": "best_aspect_2",
    "Compare remote working to working at your employer’s workplace. Select the best aspect of remote working for you - Managing my family responsibilities ; My working relationships ; Preparing for work and commuting ; My daily expenses ; My personal relationships ; My job satisfaction": "best_aspect_3",
    "Compare remote working to working at your employer’s workplace. Select the best aspect of remote working for you - The number of hours  I work ; My work-life balance ; My on-the-job learning opportunities ; Managing my personal commitments ; My opportunities to socialise ; My mental wellbeing": "best_aspect_4",
    "Compare remote working to working at your employer’s workplace. Select the best aspect of remote working for you - The number of hours  I work ; My work-life balance ; My on-the-job learning opportunities ; My daily expenses ; My personal relationships ; My job satisfaction": "best_aspect_5",
    "Compare remote working to working at your employer’s workplace. Select the best aspect of remote working for you - Managing my personal commitments ; My opportunities to socialise ; My mental wellbeing ; My daily expenses ; My personal relationships ; My job satisfaction": "best_aspect_6",
}

WORST_REMOTE_WORKING_ASPECT_RENAME_MAP = {
    "Compare remote working to working at your employer’s workplace. Select the worst aspect of remote working for you - Managing my family responsibilities ; My working relationships ; Preparing for work and commuting ; The number of hours  I work ; My work-life balance ; My on-the-job learning opportunities": "worst_aspect_1",
    "Compare remote working to working at your employer’s workplace. Select the worst aspect of remote working for you - Managing my family responsibilities ; My working relationships ; Preparing for work and commuting ; Managing my personal commitments ; My opportunities to socialise ; My mental wellbeing": "worst_aspect_2",
    "Compare remote working to working at your employer’s workplace. Select the worst aspect of remote working for you - Managing my family responsibilities ; My working relationships ; Preparing for work and commuting ; My daily expenses ; My personal relationships ; My job satisfaction": "worst_aspect_3",
    "Compare remote working to working at your employer’s workplace. Select the worst aspect of remote working for you - The number of hours  I work ; My work-life balance ; My on-the-job learning opportunities ; Managing my personal commitments ; My opportunities to socialise ; My mental wellbeing": "worst_aspect_4",
    "Compare remote working to working at your employer’s workplace. Select the worst aspect of remote working for you - The number of hours  I work ; My work-life balance ; My on-the-job learning opportunities ; My daily expenses ; My personal relationships ; My job satisfaction": "worst_aspect_5",
    "Compare remote working to working at your employer’s workplace. Select the worst aspect of remote working for you - Managing my personal commitments ; My opportunities to socialise ; My mental wellbeing ; My daily expenses ; My personal relationships ; My job satisfaction": "worst_aspect_6",
}

## Combine all batches into a single dictionary
ALL_RENAME_MAPS = {
   **CORE_DEMOGRAPHICS_RENAME_MAP,
    **REMOTE_WORK_INTENSITY_RENAME_MAP,
    **ORG_CULTURE_REMOTE_RENAME_MAP,
    **ORG_CULTURE_REMOTE_RENAME_MAP_RECENT,
    **FUTURE_EXPECTATIONS_RENAME_MAP,
    **PRODUCTIVITY_RENAME_MAP,
    **TIME_USE_OFFICE_RENAME_MAP,
    **TIME_USE_REMOTE_RENAME_MAP,
    **MOST_SIGNIFICANT_BARRIERS_RENAME_MAP,
    **LEAST_SIGNIFICANT_BARRIERS_RENAME_MAP,
    **BEST_REMOTE_WORKING_ASPECT_RENAME_MAP,
    **WORST_REMOTE_WORKING_ASPECT_RENAME_MAP
}

## Function to rename columns
def rename_columns_batch(df: pd.DataFrame, rename_map: dict) -> pd.DataFrame:
    """
    Rename columns using a mapping dictionary.
    Only renames columns that exist in the DataFrame.
    """

    existing_map = {k: v for k, v in rename_map.items() if k in df.columns}
    df = df.rename(columns=existing_map)

    return df


## Create the derived age_group feature
def add_age_group(df: pd.DataFrame, reference_year: int = 2021) -> pd.DataFrame:
    """
    Create age_group from birth_year.
    """

    if "birth_year" in df.columns:
        df["birth_year"] = pd.to_numeric(df["birth_year"], errors="coerce")
        df["age"] = reference_year - df["birth_year"]
        df["age_group"] = pd.cut(
            df["age"],
            bins=[17, 24, 34, 44, 54, 120],
            labels=["18–24", "25–34", "35–44", "45–54", "55+"]
        )
        df.drop(columns=["age"], inplace=True)

    return df

def full_pipeline(df: pd.DataFrame, rename_map: dict, reference_year: int = 2021) -> pd.DataFrame:
  """
  Apply column renaming and add age_group in a single pipeline.
  """
  df = rename_columns_batch(df, rename_map)
  df = add_age_group(df, reference_year)
  return df


In [15]:
## Copy the datasets
df_2020 = data_2020.copy()
df_2021 = data_2021.copy()

# Apply full pipeline
df_2020 = full_pipeline(df_2020, ALL_RENAME_MAPS, reference_year=2020)
df_2021 = full_pipeline(df_2021, ALL_RENAME_MAPS, reference_year=2021)

# Verify columns
print("2020 columns:", list(df_2020.columns))
print("2021 columns:", list(df_2021.columns))




2020 columns: ['response_id', 'birth_year', 'gender', 'industry_broad', 'industry_detailed', 'occupation_broad', 'occupation_detailed', 'org_size', 'is_manager', 'household_type', 'job_tenure', 'location_type', 'last_year_remote_share_actual', 'org_encourage_remote', 'org_prepared_remote', 'org_remote_common', 'org_permission_easy', 'collab_easy_remote', 'recommend_remote', 'last_year_remote_share_preferred', 'remote_share_recent_actual', 'org_encourage_remote_recent', 'org_prepared_remote_recent', 'org_remote_common_recent', 'org_permission_easy_recent', 'collab_easy_remote_recent', 'recommend_remote_recent', 'remote_share_recent_preferred', 'remote_share_future_preferred', 'future_org_encourage_remote_likelihood', 'future_org_support_changes_likelihood', 'future_remote_choice_likelihood', 'remote_productivity_vs_office', 'office_hours_commute', 'office_hours_work', 'office_hours_personal', 'office_hours_care_domestic', 'remote_hours_commute', 'remote_hours_work', 'remote_hours_person

In [16]:
import re

## Convert Likert scales to numeric values (1-5 scale)

def convert_likert_to_numeric(series: pd.Series) -> pd.Series:
    """
    Convert Likert scale text responses to numeric values.

    Standard 5-point Likert scale:
    1 = Strongly disagree (most negative)
    2 = Somewhat disagree
    3 = Neither agree nor disagree (neutral)
    4 = Somewhat agree
    5 = Strongly agree (most positive)

    Args:
        series: Pandas Series containing Likert scale text responses

    Returns:
        Pandas Series with numeric values 1-5
    """
    likert_mapping = {
        'Strongly disagree': 1,
        'Somewhat disagree': 2,
        'Neither agree nor disagree': 3,
        'Somewhat agree': 4,
        'Strongly agree': 5
    }
    return series.map(likert_mapping)

def convert_preference_to_days(series: pd.Series) -> pd.Series:
    """
    Convert preference text to numeric percentages (0-1 scale).

    Handles:
    - 'None' or 'I prefer not to work remotely' -> 0
    - 'Rarely or never' -> 0
    - 'Less than 10%' -> 0.1
    - '50% - I spent about half of my time' -> 0.5
    - '100% - I spent all of my time' -> 1
    - 'X%' -> X/100
    - Handles minor typos like '40&' -> 0.4
    """

    result = pd.Series(index=series.index, dtype=float)

    for idx, value in series.items():
        if pd.isna(value):
            result[idx] = np.nan
            continue

        value = str(value).strip().lower()  # lowercase for easier matching

        # Handle explicit None / no remote working
        if "none" in value or "not to work remotely" in value or "rarely" in value or "i would not have preferred to work remotely" in value:
            result[idx] = 0
            continue

        # Handle less than 10%
        if "less than 10" in value:
            result[idx] = 0.1
            continue

        # Handle all of the time / 100%
        if "all" in value or "100%" in value:
            result[idx] = 1
            continue

        # Handle 50% special case
        if "50%" in value or "half of my time" in value:
            result[idx] = 0.5
            continue

        # Handle other percentages (10%, 20%, 30%, 40%, 60%, 70%, 80%, 90%)
        match = re.search(r'(\d+)[%&]', value)  # % or & typo
        if match:
            result[idx] = float(match.group(1)) / 100
            continue

        # Fallback
        result[idx] = np.nan

    return result



# Get the Productivity value
def clean_productivity_value(series: pd.Series) -> pd.Series:
    """
    Extract numeric productivity values from text responses.

    Handles multiple response formats:
    - "I'm 20% more productive" -> 20
    - "They're 15% less productive" -> -15
    - "About the same" -> 0
    - "30% more productive when remote" -> 30

    Args:
        series: Pandas Series containing text responses

    Returns:
        Pandas Series with numeric values (positive for "more", negative for "less")
    """
    # Convert to string and clean special characters
    series = series.astype(str).str.replace('\x92', "'", regex=False)
    series = series.str.strip()  # Remove leading/trailing spaces

    # Initialize result series with NaN values
    result = pd.Series(index=series.index, dtype=float)

    # Handle "about the same" responses - these represent 0% change
    same_mask = series.str.contains('about.*same|same.*productive', case=False, na=False)
    result[same_mask] = 0.0

    # Extract percentages from each text response
    for idx, val in series.items():
        # Skip if already marked as "same"
        if same_mask[idx]:
            continue

        # Look for "X% more productive" pattern
        more_match = re.search(r'(\d+)%\s*more\s*productive', val, re.IGNORECASE)
        # Look for "X% less productive" pattern
        less_match = re.search(r'(\d+)%\s*less\s*productive', val, re.IGNORECASE)

        if more_match:
            # Positive value for increased productivity
            result[idx] = float(more_match.group(1))
        elif less_match:
            # Negative value for decreased productivity
            result[idx] = -float(less_match.group(1))
        else:
            # Fallback: try to find any percentage and infer direction
            pct_match = re.search(r'(\d+)%', val)
            if pct_match:
                # If "less" appears anywhere, make it negative
                if 'less' in val.lower():
                    result[idx] = -float(pct_match.group(1))
                else:
                    # Default to positive if no clear indicator
                    result[idx] = float(pct_match.group(1))

    return result



In [17]:
## Columns to convert likert to numeric

LIKERT_COLUMNS = [
    'org_encourage_remote',
    'org_prepared_remote',
    'org_remote_common',
    'org_permission_easy',
    'collab_easy_remote',
    'recommend_remote',
    'org_encourage_remote_recent',
    'org_prepared_remote_recent',
    'org_remote_common_recent',
    'org_permission_easy_recent',
    'collab_easy_remote_recent',
    'recommend_remote_recent',
    'future_org_encourage_remote_likelihood',
    'future_org_support_changes_likelihood',
    'future_remote_choice_likelihood'
]

## Columns to convert preference text to numeric percentages
PREFERENCE_COLUMNS = [
    'last_year_remote_share_actual',
    'last_year_remote_share_preferred',
    'remote_share_recent_actual',
    'remote_share_recent_preferred',
    'remote_share_future_preferred'
]

## Produtivity column
PRODUCTIVITY_COLUMN = 'remote_productivity_vs_office'


In [18]:
### Apply the pipeline to the two datasets

import numpy as np

def safe_apply_conversion(df, columns, conversion_func):
    for col in columns:
        if col in df.columns:
            df[col] = conversion_func(df[col])
        else:
            print(f"⚠️ Column not found: {col}")
    return df

try:
    # Convert Likert scales
    df_2020 = safe_apply_conversion(df_2020, LIKERT_COLUMNS, convert_likert_to_numeric)
    df_2021 = safe_apply_conversion(df_2021, LIKERT_COLUMNS, convert_likert_to_numeric)

    # Convert preference text to percentages
    df_2020 = safe_apply_conversion(df_2020, PREFERENCE_COLUMNS, convert_preference_to_days)
    df_2021 = safe_apply_conversion(df_2021, PREFERENCE_COLUMNS, convert_preference_to_days)

    # Convert productivity
    if PRODUCTIVITY_COLUMN in df_2020.columns:
        df_2020[PRODUCTIVITY_COLUMN] = clean_productivity_value(df_2020[PRODUCTIVITY_COLUMN])
    if PRODUCTIVITY_COLUMN in df_2021.columns:
        df_2021[PRODUCTIVITY_COLUMN] = clean_productivity_value(df_2021[PRODUCTIVITY_COLUMN])

    print("✅ Likert, preference, and productivity columns converted successfully")

except Exception as e:
    print("❌ Failed to convert columns")
    print(e)

✅ Likert, preference, and productivity columns converted successfully


In [19]:
# Generate comprehensive data quality report
print("\n" + "-"*80)
print("DATA QUALITY REPORT")
print("-"*80)
print("\nMissing Values:")

# Calculate missing value counts and percentages
missing_values_2020 = df_2020.isnull().sum()
missing_percentages_2020 = missing_values_2020 / len(df_2020)

missing_values_2021 = df_2021.isnull().sum()
missing_percentages_2021 = missing_values_2021 / len(df_2021)


# Only show columns that have missing values
print("\n" + "-"*80)
print("Columns with missing values:")
print("-"*80)
print("\n2020")
print("-"*80)
print(missing_values_2020[missing_values_2020 > 0])
print("\n" + "-"*80)
print("2021")
print("-"*80)
print(missing_values_2021[missing_values_2021 > 0])



--------------------------------------------------------------------------------
DATA QUALITY REPORT
--------------------------------------------------------------------------------

Missing Values:

--------------------------------------------------------------------------------
Columns with missing values:
--------------------------------------------------------------------------------

2020
--------------------------------------------------------------------------------
is_manager                       136
org_encourage_remote             250
org_prepared_remote              250
org_remote_common                250
org_permission_easy              250
collab_easy_remote               250
recommend_remote                 250
org_encourage_remote_recent       31
org_prepared_remote_recent        31
org_remote_common_recent          31
org_permission_easy_recent        31
collab_easy_remote_recent         31
recommend_remote_recent           31
remote_share_future_preferred    136
dty

In [20]:
## Save the cleaned files as CSV
df_2020.to_csv('cleaned_data_2020.csv', index=False)
df_2021.to_csv('cleaned_data_2021.csv', index=False)

## Download the CSV Files
files.download('cleaned_data_2020.csv')
files.download('cleaned_data_2021.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>